# 02 原始财务数据清洗

## 目标

在保留 `data/raw` 原始数据不变的前提下，根据上一阶段数据审计结果，
逐步建立明确、可复现的数据清洗规则，并最终输出标准化的 processed 数据。

本 Notebook 坚持：

1. 不直接修改 raw 文件；
2. 每项清洗先说明规则，再执行转换；
3. 清洗前后进行核对；
4. 对无法机械判断的问题暂不擅自删除。

## 第一阶段：股票代码标准化

`stock_code` 是公司标识符，而不是用于数学计算的数值变量。

目标格式：

- 字符串类型；
- 去除首尾空格；
- 非缺失代码统一为 6 位；
- 缺失值继续保留为缺失值；
- 不通过数值运算处理股票代码。

In [43]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

assert (PROJECT_ROOT / "pyproject.toml").exists()
assert DATA_RAW.exists()

print("Project paths initialized successfully.")

assert (PROJECT_ROOT / "pyproject.toml").exists()
assert DATA_RAW.exists()

Project paths initialized successfully.


In [2]:
raw_financials = pd.read_excel(
    DATA_RAW / "firm_financials.xlsx",
    dtype={"stock_code": "string"},
)

raw_financials.shape

(260, 11)

In [3]:
print(raw_financials["stock_code"].dtype)

raw_financials[["stock_code"]].head(30)

string


,stock_code
0,000001
1,000001
2,000001
3,000001
4,000001
5,000001
6,000002
7,2
8,000002
9,000002


In [4]:
raw_financials["stock_code"].map(repr).head(30)

0     ' 000001 '
1       '000001'
2       '000001'
3       '000001'
4       '000001'
5       '000001'
6       '000002'
7            '2'
8       '000002'
9       '000002'
10      '000002'
11      '000002'
12      '000003'
13      '000003'
14           '3'
15      '000003'
16      '000003'
17      '000003'
18      '000004'
19      '000004'
20      '000004'
21          <NA>
22      '000004'
23      '000004'
24      '000005'
25      '000005'
26      '000005'
27      '000005'
28     '000005 '
29      '000005'
Name: stock_code, dtype: str

In [5]:
financials = raw_financials.copy()

print("raw_financials shape:", raw_financials.shape)
print("financials shape:", financials.shape)

raw_financials shape: (260, 11)
financials shape: (260, 11)


In [6]:
stock_code_stripped = financials["stock_code"].str.strip()

stock_code_stripped.map(repr).head(30)

0     '000001'
1     '000001'
2     '000001'
3     '000001'
4     '000001'
5     '000001'
6     '000002'
7          '2'
8     '000002'
9     '000002'
10    '000002'
11    '000002'
12    '000003'
13    '000003'
14         '3'
15    '000003'
16    '000003'
17    '000003'
18    '000004'
19    '000004'
20    '000004'
21        <NA>
22    '000004'
23    '000004'
24    '000005'
25    '000005'
26    '000005'
27    '000005'
28    '000005'
29    '000005'
Name: stock_code, dtype: str

In [7]:
invalid_stock_code_mask = (
    stock_code_stripped.notna()
    & ~stock_code_stripped.str.fullmatch(r"\d+")
)

invalid_stock_codes = stock_code_stripped[invalid_stock_code_mask]

print("非数字股票代码数量:", len(invalid_stock_codes))
invalid_stock_codes

非数字股票代码数量: 0


Series([], Name: stock_code, dtype: string)

In [8]:
stock_code_length_counts = (
    stock_code_stripped
    .dropna()
    .str.len()
    .value_counts()
    .sort_index()
)

stock_code_length_counts

stock_code
1      2
6    257
Name: count, dtype: Int64

In [9]:
stock_code_clean = stock_code_stripped.str.zfill(6)

stock_code_clean.map(repr).head(30)

0     '000001'
1     '000001'
2     '000001'
3     '000001'
4     '000001'
5     '000001'
6     '000002'
7     '000002'
8     '000002'
9     '000002'
10    '000002'
11    '000002'
12    '000003'
13    '000003'
14    '000003'
15    '000003'
16    '000003'
17    '000003'
18    '000004'
19    '000004'
20    '000004'
21        <NA>
22    '000004'
23    '000004'
24    '000005'
25    '000005'
26    '000005'
27    '000005'
28    '000005'
29    '000005'
Name: stock_code, dtype: str

In [10]:
before_missing = raw_financials["stock_code"].isna().sum()
after_missing = stock_code_clean.isna().sum()

invalid_clean_code_mask = (
    stock_code_clean.notna()
    & ~stock_code_clean.str.fullmatch(r"\d{6}")
)

print("清洗前缺失数量:", before_missing)
print("清洗后缺失数量:", after_missing)
print("清洗后不符合6位纯数字规则的数量:", invalid_clean_code_mask.sum())

stock_code_clean.dropna().str.len().value_counts().sort_index()

清洗前缺失数量: 1
清洗后缺失数量: 1
清洗后不符合6位纯数字规则的数量: 0


stock_code
6    259
Name: count, dtype: Int64

In [11]:
same_stock_code_mask = (
    raw_financials["stock_code"].eq(stock_code_clean).fillna(False)
    | (
        raw_financials["stock_code"].isna()
        & stock_code_clean.isna()
    )
)

stock_code_changes = pd.DataFrame(
    {
        "before": raw_financials["stock_code"].map(repr),
        "after": stock_code_clean.map(repr),
    }
)

stock_code_changes = stock_code_changes.loc[~same_stock_code_mask]

print("实际发生变化的记录数:", len(stock_code_changes))
stock_code_changes

实际发生变化的记录数: 5


,before,after
0,' 000001 ','000001'
7,'2','000002'
14,'3','000003'
28,'000005 ','000005'
240,' 000001 ','000001'


In [12]:
assert before_missing == after_missing
assert invalid_clean_code_mask.sum() == 0
assert stock_code_clean.dropna().str.len().eq(6).all()

financials["stock_code"] = stock_code_clean

print("stock_code 标准化完成")
print(financials["stock_code"].dtype)

stock_code 标准化完成
string


In [13]:
print("year dtype:", financials["year"].dtype)

financials["year"].map(type).value_counts()

year dtype: object


year
<class 'int'>    250
<class 'str'>     10
Name: count, dtype: int64

In [14]:
financials["year"].value_counts(dropna=False).sort_index(key=lambda x: x.astype(str))

year
2020     43
2021     43
2022     39
2022      4
2023     38
2023年     6
2024     44
2025     43
Name: count, dtype: int64

In [15]:
year_text = financials["year"].astype("string").str.strip()

year_text.value_counts(dropna=False).sort_index()

year
2020     43
2021     43
2022     43
2023     38
2023年     6
2024     44
2025     43
Name: count, dtype: int64[pyarrow]

In [16]:
invalid_year_mask = (
    year_text.notna()
    & ~year_text.str.fullmatch(r"\d{4}")
)

invalid_years = year_text[invalid_year_mask]

print("非标准年份记录数:", len(invalid_years))
invalid_years.value_counts(dropna=False)

非标准年份记录数: 6


year
2023年    6
Name: count, dtype: int64[pyarrow]

In [17]:
year_clean_text = year_text.str.replace(r"年$", "", regex=True)

year_clean_text.value_counts(dropna=False).sort_index()

year
2020    43
2021    43
2022    43
2023    44
2024    44
2025    43
Name: count, dtype: int64[pyarrow]

In [18]:
invalid_year_after_rule = (
    year_clean_text.notna()
    & ~year_clean_text.str.fullmatch(r"\d{4}")
)

print(
    "应用年份格式规则后，仍非4位纯数字的记录数:",
    invalid_year_after_rule.sum(),
)

year_clean_text[invalid_year_after_rule]

应用年份格式规则后，仍非4位纯数字的记录数: 0


Series([], Name: year, dtype: string)

In [19]:
year_clean = pd.to_numeric(
    year_clean_text,
    errors="raise",
).astype("Int64")

print("清洗后 year dtype:", year_clean.dtype)

year_clean.value_counts(dropna=False).sort_index()

清洗后 year dtype: Int64


year
2020    43
2021    43
2022    43
2023    44
2024    44
2025    43
Name: count, dtype: Int64

In [20]:
year_missing_before = raw_financials["year"].isna().sum()
year_missing_after = year_clean.isna().sum()

year_out_of_range_mask = (
    year_clean.notna()
    & ~year_clean.between(2020, 2025)
)

print("清洗前年份缺失数量:", year_missing_before)
print("清洗后年份缺失数量:", year_missing_after)
print("超出 2020-2025 范围的年份数量:", year_out_of_range_mask.sum())
print("最小年份:", year_clean.min())
print("最大年份:", year_clean.max())

清洗前年份缺失数量: 0
清洗后年份缺失数量: 0
超出 2020-2025 范围的年份数量: 0
最小年份: 2020
最大年份: 2025


In [21]:
year_before_text = raw_financials["year"].astype("string")

year_same_mask = (
    year_before_text.eq(year_clean.astype("string")).fillna(False)
    | (
        raw_financials["year"].isna()
        & year_clean.isna()
    )
)

year_changes = pd.DataFrame(
    {
        "before": raw_financials["year"].map(repr),
        "after": year_clean.map(repr),
    }
).loc[~year_same_mask]

print("年份表示实际发生变化的记录数:", len(year_changes))
year_changes

年份表示实际发生变化的记录数: 6


,before,after
3,'2023年',2023
9,'2023年',2023
15,'2023年',2023
21,'2023年',2023
27,'2023年',2023
251,'2023年',2023


In [22]:
assert invalid_year_after_rule.sum() == 0
assert year_missing_before == year_missing_after
assert year_out_of_range_mask.sum() == 0
assert year_clean.dropna().between(2020, 2025).all()

financials["year"] = year_clean

print("year 标准化完成")
print("year dtype:", financials["year"].dtype)

year 标准化完成
year dtype: Int64


## 第二阶段：firm-year 主键重复检查

在完成 `stock_code` 和 `year` 标准化后，
重新检查 `stock_code + year` 是否能够唯一标识一条公司年度记录。

本阶段只识别和分类重复问题，不直接删除记录。

In [23]:
key_cols = ["stock_code", "year"]

firm_year_duplicate_mask = financials.duplicated(
    subset=key_cols,
    keep=False,
)

duplicate_firm_year_rows = financials.loc[
    firm_year_duplicate_mask
].copy()

print(
    "位于重复 firm-year 组中的记录数:",
    firm_year_duplicate_mask.sum(),
)

duplicate_group_count = (
    duplicate_firm_year_rows
    .groupby(key_cols, dropna=False)
    .ngroups
)

print(
    "重复 firm-year 组数:",
    duplicate_group_count,
)

位于重复 firm-year 组中的记录数: 40
重复 firm-year 组数: 20


In [24]:
duplicate_group_sizes = (
    duplicate_firm_year_rows
    .groupby(key_cols, dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values(
        ["row_count", "stock_code", "year"],
        ascending=[False, True, True],
    )
)

duplicate_group_sizes

,stock_code,year,row_count
0,000001,2020,2
1,000002,2024,2
2,000003,2025,2
3,000005,2023,2
4,000006,2024,2
5,000008,2022,2
6,000009,2023,2
7,000011,2021,2
8,000012,2022,2
9,000014,2020,2


In [25]:
duplicate_firm_year_rows = (
    duplicate_firm_year_rows
    .sort_values(key_cols)
)

duplicate_firm_year_rows[
    [
        "stock_code",
        "company_name",
        "year",
        "total_assets",
        "total_liabilities",
        "revenue",
        "net_profit",
        "cash",
        "rd_expense",
        "roe",
        "employees",
    ]
]

,stock_code,company_name,year,total_assets,total_liabilities,revenue,net_profit,cash,rd_expense,roe,employees
0,000001,华辰科技股份有限公司,2020,44813001602,23908847993,8788852319,8.233325e+08,4.322096e+09,655221236,0.0394,14725.0
240,000001,华辰科技股份有限公司,2020,44813001602,23908847993,8788852319,8.233325e+08,4.322096e+09,655221236,0.0394,14725.0
10,000002,新岳科技股份有限公司,2024,9274052273,2248802709,44523089214,1.471324e+10,1.739329e+09,2588058155,2.0943,36975.0
250,000002,新岳科技股份有限公司,2024,9274052273,2248802709,44523089214,1.500893e+09,1.739329e+09,2588058155,2.0943,36975.0
17,000003,海川科技股份有限公司,2025,51903132738,21094446603,6197206185,1.456729e+08,1.501682e+10,367316599,0.0047,17290.0
241,000003,海川科技股份有限公司,2025,51903132738,21094446603,6197206185,1.456729e+08,1.501682e+10,367316599,0.0047,17290.0
27,000005,宏远科技股份有限公司,2023,33191213417,20487864880,25711723804,2.855787e+09,3.907137e+09,1678145406,0.2248,3315.0
251,000005,宏远科技股份有限公司,2023,33191213417,20487864880,25711723804,1.766409e+09,3.907137e+09,1678145406,0.2248,3315.0
34,000006,智恒科技股份有限公司,2024,24192980006,8900245804,24585101604,4.884060e+09,5.243925e+09,390759665,0.3194,44831.0
242,000006,智恒科技股份有限公司,2024,24192980006,8900245804,24585101604,4.884060e+09,5.243925e+09,390759665,0.3194,44831.0


In [26]:
print(
    "完全重复的后续记录数:",
    financials.duplicated(keep="first").sum(),
)

print(
    "位于完全重复组中的全部记录数:",
    financials.duplicated(keep=False).sum(),
)

完全重复的后续记录数: 10
位于完全重复组中的全部记录数: 20


## 第三阶段：重复 firm-year 分类

重复的 `stock_code + year` 组合需要进一步区分：

- **完全重复**：同一 firm-year 的所有字段完全一致；
- **冲突重复**：firm-year 相同，但一个或多个变量取值不同。

完全重复通常可以安全去除多余副本；
冲突重复则不能机械删除，需要先识别冲突字段并决定处理规则。

In [27]:
non_key_cols = [
    col
    for col in financials.columns
    if col not in key_cols
]

group_variation = (
    duplicate_firm_year_rows
    .groupby(key_cols, dropna=False)[non_key_cols]
    .nunique(dropna=False)
)

group_variation

,,company_name,total_assets,total_liabilities,revenue,net_profit,cash,rd_expense,roe,employees
stock_code,year,,,,,,,,,
000001,2020,1,1,1,1,1,1,1,1,1
000002,2024,1,1,1,1,2,1,1,1,1
000003,2025,1,1,1,1,1,1,1,1,1
000005,2023,1,1,1,1,2,1,1,1,1
000006,2024,1,1,1,1,1,1,1,1,1
000008,2022,1,1,1,1,2,1,1,1,1
000009,2023,1,1,1,1,1,1,1,1,1
000011,2021,1,1,1,1,2,1,1,1,1
000012,2022,1,1,1,1,1,1,1,1,1


In [28]:
duplicate_group_classification = pd.DataFrame(
    index=group_variation.index
)

duplicate_group_classification["max_distinct_values"] = (
    group_variation.max(axis=1)
)

duplicate_group_classification["duplicate_type"] = (
    duplicate_group_classification["max_distinct_values"]
    .eq(1)
    .map(
        {
            True: "exact",
            False: "conflict",
        }
    )
)

duplicate_group_classification.reset_index()

,stock_code,year,max_distinct_values,duplicate_type
0,000001,2020,1,exact
1,000002,2024,2,conflict
2,000003,2025,1,exact
3,000005,2023,2,conflict
4,000006,2024,1,exact
5,000008,2022,2,conflict
6,000009,2023,1,exact
7,000011,2021,2,conflict
8,000012,2022,1,exact
9,000014,2020,2,conflict


In [29]:
duplicate_group_classification[
    "duplicate_type"
].value_counts()

duplicate_type
exact       10
conflict    10
Name: count, dtype: int64

In [30]:
conflict_group_mask = (
    duplicate_group_classification["duplicate_type"]
    == "conflict"
)

conflict_variation = group_variation.loc[
    conflict_group_mask
]

conflict_summary = (
    conflict_variation
    .gt(1)
    .apply(
        lambda row: ", ".join(
            row.index[row].tolist()
        ),
        axis=1,
    )
    .rename("different_columns")
    .reset_index()
)

conflict_summary

,stock_code,year,different_columns
0,000002,2024,net_profit
1,000005,2023,net_profit
2,000008,2022,net_profit
3,000011,2021,net_profit
4,000014,2020,net_profit
5,000016,2025,net_profit
6,000019,2024,net_profit
7,000022,2023,net_profit
8,000025,2022,net_profit
9,000028,2021,net_profit


In [31]:
duplicate_rows_classified = (
    duplicate_firm_year_rows
    .merge(
        duplicate_group_classification[
            ["duplicate_type"]
        ].reset_index(),
        on=key_cols,
        how="left",
        validate="many_to_one",
    )
)

duplicate_rows_classified[
    "duplicate_type"
].value_counts()

duplicate_type
exact       20
conflict    20
Name: count, dtype: int64

## 第四阶段：安全去除完全重复记录

在 20 个重复 firm-year 组中：

- 10 个属于完全重复；
- 10 个属于冲突重复；
- 冲突重复目前全部只在 `net_profit` 上存在不同取值。

本阶段只删除完全重复记录中的多余副本。

冲突重复记录全部保留，后续单独制定处理规则。

In [32]:
exact_duplicate_later_mask = financials.duplicated(
    keep="first"
)

rows_to_remove = financials.loc[
    exact_duplicate_later_mask
].copy()

print("准备删除的完全重复记录数:", len(rows_to_remove))

准备删除的完全重复记录数: 10


In [33]:
rows_to_remove_check = rows_to_remove.merge(
    duplicate_group_classification[
        ["duplicate_type"]
    ].reset_index(),
    on=key_cols,
    how="left",
    validate="many_to_one",
)

print(
    rows_to_remove_check[
        "duplicate_type"
    ].value_counts(dropna=False)
)

rows_to_remove_check[
    [
        "stock_code",
        "year",
        "duplicate_type",
    ]
]

duplicate_type
exact    10
Name: count, dtype: int64


,stock_code,year,duplicate_type
0,000001,2020,exact
1,000003,2025,exact
2,000006,2024,exact
3,000009,2023,exact
4,000012,2022,exact
5,000015,2021,exact
6,000018,2020,exact
7,000020,2025,exact
8,000023,2024,exact
9,000026,2023,exact


In [34]:
financials_after_exact_dedup = financials.loc[
    ~exact_duplicate_later_mask
].copy()

print("删除前记录数:", len(financials))
print("删除后记录数:", len(financials_after_exact_dedup))
print(
    "实际减少记录数:",
    len(financials) - len(financials_after_exact_dedup),
)

删除前记录数: 260
删除后记录数: 250
实际减少记录数: 10


In [35]:
remaining_firm_year_duplicate_mask = (
    financials_after_exact_dedup.duplicated(
        subset=key_cols,
        keep=False,
    )
)

remaining_duplicate_rows = (
    financials_after_exact_dedup.loc[
        remaining_firm_year_duplicate_mask
    ].copy()
)

remaining_duplicate_group_count = (
    remaining_duplicate_rows
    .groupby(key_cols, dropna=False)
    .ngroups
)

remaining_exact_duplicate_count = (
    financials_after_exact_dedup
    .duplicated(keep="first")
    .sum()
)

print(
    "剩余重复 firm-year 记录数:",
    remaining_firm_year_duplicate_mask.sum(),
)

print(
    "剩余重复 firm-year 组数:",
    remaining_duplicate_group_count,
)

print(
    "剩余完全重复后续记录数:",
    remaining_exact_duplicate_count,
)

剩余重复 firm-year 记录数: 20
剩余重复 firm-year 组数: 10
剩余完全重复后续记录数: 0


In [36]:
conflict_keys_before = set(
    map(
        tuple,
        duplicate_group_classification
        .query("duplicate_type == 'conflict'")
        .reset_index()[key_cols]
        .to_numpy(),
    )
)

remaining_duplicate_keys = set(
    map(
        tuple,
        remaining_duplicate_rows[
            key_cols
        ]
        .drop_duplicates()
        .to_numpy(),
    )
)

print("原 conflict 组数:", len(conflict_keys_before))
print("删除后剩余重复组数:", len(remaining_duplicate_keys))

print(
    "剩余重复组是否恰好等于原 conflict 组:",
    conflict_keys_before == remaining_duplicate_keys,
)

原 conflict 组数: 10
删除后剩余重复组数: 10
剩余重复组是否恰好等于原 conflict 组: True


In [37]:
assert len(rows_to_remove) == 10

assert (
    rows_to_remove_check["duplicate_type"]
    .eq("exact")
    .all()
)

assert len(financials_after_exact_dedup) == 250

assert remaining_exact_duplicate_count == 0

assert (
    remaining_firm_year_duplicate_mask.sum()
    == 20
)

assert remaining_duplicate_group_count == 10

assert (
    conflict_keys_before
    == remaining_duplicate_keys
)

financials = financials_after_exact_dedup

print("完全重复记录清理完成")
print("当前记录数:", len(financials))

完全重复记录清理完成
当前记录数: 250


## 第五阶段：财务数值字段审计

完成主键标准化和完全重复去重后，开始检查财务数值字段。

本阶段先识别：
- 哪些列已经是数值类型；
- 哪些列因为特殊字符串而变成 object；
- 是否存在逗号、百分号、伪缺失值等格式问题。

暂不进行实际转换。

In [38]:
numeric_candidate_cols = [
    "total_assets",
    "total_liabilities",
    "revenue",
    "net_profit",
    "cash",
    "rd_expense",
    "roe",
    "employees",
]

financials[numeric_candidate_cols].dtypes

total_assets          object
total_liabilities     object
revenue               object
net_profit           float64
cash                 float64
rd_expense            object
roe                   object
employees            float64
dtype: object

In [39]:
for col in numeric_candidate_cols:
    string_mask = financials[col].map(
        lambda x: isinstance(x, str)
    )

    string_values = financials.loc[
        string_mask,
        col,
    ]

    print("=" * 60)
    print("变量:", col)
    print("字符串记录数:", len(string_values))

    if len(string_values) > 0:
        print(
            string_values
            .value_counts(dropna=False)
            .head(20)
        )

变量: total_assets
字符串记录数: 1
total_assets
31,241,347,993    1
Name: count, dtype: int64
变量: total_liabilities
字符串记录数: 1
total_liabilities
-    1
Name: count, dtype: int64
变量: revenue
字符串记录数: 1
revenue
15,335,659,776    1
Name: count, dtype: int64
变量: net_profit
字符串记录数: 0
变量: cash
字符串记录数: 0
变量: rd_expense
字符串记录数: 1
rd_expense
-    1
Name: count, dtype: int64
变量: roe
字符串记录数: 2
roe
13.5%    1
7.25%    1
Name: count, dtype: int64
变量: employees
字符串记录数: 0


In [40]:
numeric_missing_summary = pd.DataFrame(
    {
        "dtype": financials[
            numeric_candidate_cols
        ].dtypes.astype(str),

        "real_missing": financials[
            numeric_candidate_cols
        ].isna().sum(),
    }
)

numeric_missing_summary

,dtype,real_missing
total_assets,object,0
total_liabilities,object,0
revenue,object,0
net_profit,float64,1
cash,float64,1
rd_expense,object,0
roe,object,0
employees,float64,1


In [41]:
for col in ["rd_expense", "roe"]:
    string_mask = financials[col].map(
        lambda x: isinstance(x, str)
    )

    string_values = financials.loc[
        string_mask,
        col,
    ]

    print("=" * 50)
    print("变量:", col)
    print("字符串记录数:", len(string_values))
    print("字符串取值及数量:")
    print(string_values.value_counts(dropna=False))

变量: rd_expense
字符串记录数: 1
字符串取值及数量:
rd_expense
-    1
Name: count, dtype: int64
变量: roe
字符串记录数: 2
字符串取值及数量:
roe
13.5%    1
7.25%    1
Name: count, dtype: int64


In [42]:
roe_numeric_mask = financials["roe"].map(
    lambda x: isinstance(x, (int, float))
)

roe_numeric = financials.loc[
    roe_numeric_mask,
    "roe",
]

print("数值型 ROE 记录数:", len(roe_numeric))
print("最小值:", roe_numeric.min())
print("最大值:", roe_numeric.max())

print("\n绝对值最大的 15 条数值型 ROE:")
print(
    roe_numeric
    .abs()
    .sort_values(ascending=False)
    .head(15)
)

数值型 ROE 记录数: 248
最小值: -1.9169
最大值: 38.593

绝对值最大的 15 条数值型 ROE:
91     38.593
58       13.5
211    5.8785
139    4.9073
40     3.3773
35     2.1835
10     2.0943
250    2.0943
42     1.9169
130    1.7754
236    1.6603
252    1.5301
44     1.5301
205    1.3511
111    1.3407
Name: roe, dtype: object


## 阶段性结论：2026-09-17

本阶段已完成财务面板训练数据的第一轮正式清洗与数值字段审计。

已完成：

- `stock_code` 标准化为 6 位字符串，并保留原有缺失；
- `year` 标准化为 pandas `Int64` 年份；
- 识别 20 个重复 firm-year 组；
- 将重复组区分为 10 个完全重复组和 10 个冲突重复组；
- 安全删除 10 条完全重复的多余副本，记录数由 260 条降至 250 条；
- 保留全部冲突 firm-year，不进行无依据删除；
- 已确认剩余 10 个冲突组均只在 `net_profit` 上存在不同取值；
- 完成主要财务数值字段的格式审计。

当前已识别的数值字段问题包括：

- `total_assets`：存在千位逗号格式；
- `revenue`：存在千位逗号格式；
- `total_liabilities`：存在 `"-"` 伪缺失值；
- `rd_expense`：存在 `"-"` 伪缺失值；
- `net_profit`、`cash`、`employees`：存在真实缺失值；
- `roe`：同时存在数值形式与百分号字符串形式，并存在需要进一步判断的异常或尺度问题。

本阶段暂不强行处理冲突 firm-year 和 ROE，后续将在明确规则后继续清洗。

原始数据文件始终保持不修改。

## 第六阶段：基础数值字段标准化

根据前一阶段审计结果，对规则已经明确的数值字段进行标准化：

- `total_assets`：去除千位逗号；
- `revenue`：去除千位逗号；
- `total_liabilities`：将 `"-"` 识别为缺失值；
- `rd_expense`：将 `"-"` 识别为缺失值。

本阶段不处理 `roe`，因为其同时存在小数、百分号字符串和疑似尺度异常，需要单独判断。

In [44]:
basic_numeric_cols = [
    "total_assets",
    "total_liabilities",
    "revenue",
    "rd_expense",
]

basic_numeric_clean = {}

for col in basic_numeric_cols:
    text = (
        financials[col]
        .astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
        .replace("-", pd.NA)
    )

    basic_numeric_clean[col] = (
        pd.to_numeric(text, errors="raise")
        .astype("Float64")
    )

print("基础数值字段候选转换完成")

基础数值字段候选转换完成


In [45]:
basic_numeric_validation = []

for col in basic_numeric_cols:
    before_missing = financials[col].isna().sum()
    after_missing = basic_numeric_clean[col].isna().sum()

    basic_numeric_validation.append(
        {
            "column": col,
            "before_missing": before_missing,
            "after_missing": after_missing,
            "new_missing": after_missing - before_missing,
        }
    )

basic_numeric_validation = pd.DataFrame(
    basic_numeric_validation
)

basic_numeric_validation

,column,before_missing,after_missing,new_missing
0,total_assets,0,0,0
1,total_liabilities,0,1,1
2,revenue,0,0,0
3,rd_expense,0,1,1


In [47]:
for col in basic_numeric_cols:
    original_string_mask = financials[col].map(
        lambda x: isinstance(x, str)
    )

    changes = pd.DataFrame(
        {
            "before": financials.loc[
                original_string_mask, col
            ].map(repr),
            "after": basic_numeric_clean[col].loc[
                original_string_mask
            ].map(repr),
        }
    )

    print("=" * 60)
    print(col)
    print(changes)

total_assets
             before          after
8  '31,241,347,993'  31241347993.0
total_liabilities
   before after
29    '-'   nan
revenue
              before          after
47  '15,335,659,776'  15335659776.0
rd_expense
    before after
115    '-'   nan


In [48]:
assert basic_numeric_validation.loc[
    basic_numeric_validation["column"] == "total_assets",
    "new_missing",
].iloc[0] == 0

assert basic_numeric_validation.loc[
    basic_numeric_validation["column"] == "revenue",
    "new_missing",
].iloc[0] == 0

assert basic_numeric_validation.loc[
    basic_numeric_validation["column"] == "total_liabilities",
    "new_missing",
].iloc[0] == 1

assert basic_numeric_validation.loc[
    basic_numeric_validation["column"] == "rd_expense",
    "new_missing",
].iloc[0] == 1

for col in basic_numeric_cols:
    financials[col] = basic_numeric_clean[col]

print(financials[basic_numeric_cols].dtypes)

total_assets         Float64
total_liabilities    Float64
revenue              Float64
rd_expense           Float64
dtype: object


In [49]:
conflict_keys_df = (
    duplicate_group_classification
    .query("duplicate_type == 'conflict'")
    .reset_index()[key_cols]
)

conflict_key_index = pd.MultiIndex.from_frame(
    conflict_keys_df
)

current_key_index = pd.MultiIndex.from_frame(
    financials[key_cols]
)

non_conflict_mask = ~current_key_index.isin(
    conflict_key_index
)

roe_check_sample = financials.loc[
    non_conflict_mask
].copy()

print("当前总记录数:", len(financials))
print("排除 conflict 后记录数:", len(roe_check_sample))

当前总记录数: 250
排除 conflict 后记录数: 230


In [50]:
roe_check_sample["equity_implied"] = (
    roe_check_sample["total_assets"]
    - roe_check_sample["total_liabilities"]
)

valid_roe_check_mask = (
    roe_check_sample["equity_implied"].notna()
    & roe_check_sample["net_profit"].notna()
    & roe_check_sample["equity_implied"].ne(0)
)

roe_check_sample["roe_implied"] = pd.Series(
    pd.NA,
    index=roe_check_sample.index,
    dtype="Float64",
)

roe_check_sample.loc[
    valid_roe_check_mask,
    "roe_implied",
] = (
    roe_check_sample.loc[
        valid_roe_check_mask,
        "net_profit",
    ]
    / roe_check_sample.loc[
        valid_roe_check_mask,
        "equity_implied",
    ]
)

print("可计算隐含 ROE 的记录数:", valid_roe_check_mask.sum())
print("无法计算隐含 ROE 的记录数:", (~valid_roe_check_mask).sum())

可计算隐含 ROE 的记录数: 228
无法计算隐含 ROE 的记录数: 2


In [51]:
roe_text = (
    roe_check_sample["roe"]
    .astype("string")
    .str.strip()
)

roe_percent_string_mask = (
    roe_text.str.endswith("%", na=False)
)

roe_direct_numeric = pd.to_numeric(
    roe_text.where(~roe_percent_string_mask),
    errors="coerce",
).astype("Float64")

roe_percent_decimal = pd.Series(
    pd.NA,
    index=roe_check_sample.index,
    dtype="Float64",
)

roe_percent_decimal.loc[
    roe_percent_string_mask
] = (
    pd.to_numeric(
        roe_text.loc[
            roe_percent_string_mask
        ].str.replace("%", "", regex=False),
        errors="raise",
    ).astype("Float64")
    / 100
)

special_roe_mask = (
    roe_percent_string_mask
    | roe_direct_numeric.abs().gt(5).fillna(False)
)

roe_special_cases = roe_check_sample.loc[
    special_roe_mask,
    [
        "stock_code",
        "year",
        "total_assets",
        "total_liabilities",
        "net_profit",
        "roe",
        "equity_implied",
        "roe_implied",
    ],
].copy()

roe_special_cases["roe_as_decimal"] = (
    roe_direct_numeric.loc[special_roe_mask]
)

roe_special_cases["roe_if_divided_by_100"] = (
    roe_direct_numeric.loc[special_roe_mask]
    / 100
)

roe_special_cases["percent_string_decimal"] = (
    roe_percent_decimal.loc[special_roe_mask]
)

roe_special_cases

,stock_code,year,total_assets,total_liabilities,net_profit,roe,equity_implied,roe_implied,roe_as_decimal,roe_if_divided_by_100,percent_string_decimal
14,000003,2022,37487571181.0,18849873227.0,2.288218e+08,13.5%,18637697954.0,0.012277,<NA>,<NA>,0.135
58,000010,2024,11599589712.0,6146176840.0,1.205854e+09,13.5,5453412872.0,0.221119,13.5,0.135,<NA>
91,000016,2021,825487570.0,634028098.0,7.389000e+09,38.593,191459472.0,38.593026,38.593,0.38593,<NA>
103,000018,2021,24257838029.0,19190115718.0,2.048042e+08,7.25%,5067722311.0,0.040413,<NA>,<NA>,0.0725
211,000036,2021,2434203929.0,1377288331.0,6.213027e+09,5.8785,1056915598.0,5.878451,5.8785,0.058785,<NA>


## 第七阶段：ROE 口径核验与修复

审计发现 `roe` 同时包含普通数值、百分号字符串及极端数值。

通过与财务变量关系核验，并结合本模拟训练数据的生成规则确认：

`roe = net_profit / (total_assets - total_liabilities)`

其中 3 条记录被人为注入了错误 ROE 表示，不能仅通过删除百分号或统一除以 100 修复。

因此本训练数据中对已确认异常的 ROE 使用财务关系重新计算。

同时保留能够与财务关系吻合的极端 ROE，不因为数值较大而机械删除或缩放。

注意：该修复规则依赖于本模拟数据已知的字段定义。真实科研中必须先确认数据来源与变量口径，不能直接照搬。

In [52]:
roe_numeric_candidate = pd.to_numeric(
    financials["roe"]
    .astype("string")
    .str.strip()
    .str.replace("%", "", regex=False),
    errors="coerce",
).astype("Float64")

roe_formula_candidate = (
    financials["net_profit"]
    / (
        financials["total_assets"]
        - financials["total_liabilities"]
    )
).astype("Float64")

roe_formula_candidate = roe_formula_candidate.round(4)

In [53]:
roe_difference = (
    roe_numeric_candidate
    - roe_formula_candidate
).abs()

roe_mismatch_mask = (
    roe_numeric_candidate.notna()
    & roe_formula_candidate.notna()
    & roe_difference.gt(0.0001)
)

roe_mismatches = financials.loc[
    roe_mismatch_mask,
    [
        "stock_code",
        "year",
        "net_profit",
        "total_assets",
        "total_liabilities",
        "roe",
    ],
].copy()

roe_mismatches["roe_formula"] = (
    roe_formula_candidate.loc[roe_mismatch_mask]
)

roe_mismatches["absolute_difference"] = (
    roe_difference.loc[roe_mismatch_mask]
)

print("ROE 与已知计算规则不一致的记录数:", len(roe_mismatches))

roe_mismatches

ROE 与已知计算规则不一致的记录数: 15


,stock_code,year,net_profit,total_assets,total_liabilities,roe,roe_formula,absolute_difference
14,000003,2022,2.288218e+08,37487571181.0,18849873227.0,13.5%,0.0123,13.4877
58,000010,2024,1.205854e+09,11599589712.0,6146176840.0,13.5,0.2211,13.2789
103,000018,2021,2.048042e+08,24257838029.0,19190115718.0,7.25%,0.0404,7.2096
166,000028,2024,-1.937456e+09,9999999999999.0,5517209077.0,-0.1328,-0.0002,0.1326
177,000030,2023,-8.888889e+09,39215609364.0,7630578962.0,0.0306,-0.2814,0.312
250,000002,2024,1.500893e+09,9274052273.0,2248802709.0,2.0943,0.2136,1.8807
251,000005,2023,1.766409e+09,33191213417.0,20487864880.0,0.2248,0.1391,0.0857
252,000008,2022,1.573933e+09,2729598687.0,701324636.0,-1.5301,0.776,2.3061
253,000011,2021,-3.872496e+08,11495096003.0,5363492081.0,0.0926,-0.0632,0.1558
254,000014,2020,1.210936e+09,41000762909.0,15785504488.0,0.0324,0.048,0.0156


In [54]:
roe_non_conflict_mismatch_mask = (
    non_conflict_mask
    & roe_numeric_candidate.notna()
    & roe_formula_candidate.notna()
    & roe_difference.gt(0.0001)
)

roe_non_conflict_mismatches = financials.loc[
    roe_non_conflict_mismatch_mask,
    [
        "stock_code",
        "year",
        "roe",
        "net_profit",
        "total_assets",
        "total_liabilities",
    ],
].copy()

roe_non_conflict_mismatches["roe_formula"] = (
    roe_formula_candidate.loc[
        roe_non_conflict_mismatch_mask
    ]
)

print(
    "排除 net_profit 冲突组后，"
    "ROE 自身不一致记录数:",
    len(roe_non_conflict_mismatches),
)

roe_non_conflict_mismatches

排除 net_profit 冲突组后，ROE 自身不一致记录数: 5


,stock_code,year,roe,net_profit,total_assets,total_liabilities,roe_formula
14,000003,2022,13.5%,2.288218e+08,37487571181.0,18849873227.0,0.0123
58,000010,2024,13.5,1.205854e+09,11599589712.0,6146176840.0,0.2211
103,000018,2021,7.25%,2.048042e+08,24257838029.0,19190115718.0,0.0404
166,000028,2024,-0.1328,-1.937456e+09,9999999999999.0,5517209077.0,-0.0002
177,000030,2023,0.0306,-8.888889e+09,39215609364.0,7630578962.0,-0.2814


In [55]:
roe_clean = roe_numeric_candidate.copy()

roe_clean.loc[
    roe_non_conflict_mismatch_mask
] = (
    roe_formula_candidate.loc[
        roe_non_conflict_mismatch_mask
    ]
)

In [56]:
roe_repairs = pd.DataFrame(
    {
        "before": financials.loc[
            roe_non_conflict_mismatch_mask,
            "roe",
        ].map(repr),

        "after": roe_clean.loc[
            roe_non_conflict_mismatch_mask
        ],
    }
)

roe_repairs

,before,after
14,'13.5%',0.0123
58,13.5,0.2211
103,'7.25%',0.0404
166,-0.1328,-0.0002
177,0.0306,-0.2814


In [57]:
extreme_roe_check = pd.DataFrame(
    {
        "before": financials["roe"],
        "after": roe_clean,
        "formula": roe_formula_candidate,
    }
)

extreme_roe_check = extreme_roe_check.loc[
    roe_clean.abs().gt(5).fillna(False)
]

extreme_roe_check

,before,after,formula
91,38.593,38.593,38.593
211,5.8785,5.8785,5.8785


In [58]:
known_upstream_anomaly_mask = (
    financials["total_assets"].eq(9_999_999_999_999)
    | financials["net_profit"].eq(-8_888_888_888)
)

print(
    "已知上游财务变量异常记录数:",
    known_upstream_anomaly_mask.sum(),
)

financials.loc[
    known_upstream_anomaly_mask,
    [
        "stock_code",
        "year",
        "total_assets",
        "net_profit",
        "roe",
    ],
]

已知上游财务变量异常记录数: 2


,stock_code,year,total_assets,net_profit,roe
166,000028,2024,9999999999999.0,-1.937456e+09,-0.1328
177,000030,2023,39215609364.0,-8.888889e+09,0.0306


In [59]:
roe_direct_error_mask = (
    non_conflict_mask
    & ~known_upstream_anomaly_mask
    & roe_numeric_candidate.notna()
    & roe_formula_candidate.notna()
    & roe_difference.gt(0.0001)
)

roe_direct_errors = financials.loc[
    roe_direct_error_mask,
    [
        "stock_code",
        "year",
        "roe",
        "net_profit",
        "total_assets",
        "total_liabilities",
    ],
].copy()

roe_direct_errors["roe_formula"] = (
    roe_formula_candidate.loc[
        roe_direct_error_mask
    ]
)

print(
    "排除 conflict 和上游已知异常后，"
    "确认的 ROE 异常记录数:",
    len(roe_direct_errors),
)

roe_direct_errors

排除 conflict 和上游已知异常后，确认的 ROE 异常记录数: 3


,stock_code,year,roe,net_profit,total_assets,total_liabilities,roe_formula
14,000003,2022,13.5%,2.288218e+08,37487571181.0,18849873227.0,0.0123
58,000010,2024,13.5,1.205854e+09,11599589712.0,6146176840.0,0.2211
103,000018,2021,7.25%,2.048042e+08,24257838029.0,19190115718.0,0.0404


In [60]:
roe_clean = roe_numeric_candidate.copy()

roe_clean.loc[
    roe_direct_error_mask
] = (
    roe_formula_candidate.loc[
        roe_direct_error_mask
    ]
)

roe_repairs = pd.DataFrame(
    {
        "before": financials.loc[
            roe_direct_error_mask,
            "roe",
        ].map(repr),
        "after": roe_clean.loc[
            roe_direct_error_mask
        ],
    }
)

roe_repairs

,before,after
14,'13.5%',0.0123
58,13.5,0.2211
103,'7.25%',0.0404


In [61]:
roe_issue_summary = pd.DataFrame(
    {
        "category": [
            "net_profit conflict",
            "known upstream anomaly",
            "direct ROE error",
        ],
        "row_count": [
            (
                roe_mismatch_mask
                & ~non_conflict_mask
            ).sum(),
            (
                roe_mismatch_mask
                & non_conflict_mask
                & known_upstream_anomaly_mask
            ).sum(),
            roe_direct_error_mask.sum(),
        ],
    }
)

roe_issue_summary

,category,row_count
0,net_profit conflict,10
1,known upstream anomaly,2
2,direct ROE error,3


In [62]:
extreme_roe_check = pd.DataFrame(
    {
        "before": financials["roe"],
        "after": roe_clean,
        "formula": roe_formula_candidate,
    }
)

extreme_roe_check = extreme_roe_check.loc[
    roe_clean.abs().gt(5).fillna(False)
]

extreme_roe_check

,before,after,formula
91,38.593,38.593,38.593
211,5.8785,5.8785,5.8785


In [63]:
assert known_upstream_anomaly_mask.sum() == 2
assert roe_direct_error_mask.sum() == 3

assert (
    roe_issue_summary["row_count"].sum()
    == roe_mismatch_mask.sum()
)

assert (
    extreme_roe_check["after"]
    .sub(extreme_roe_check["formula"])
    .abs()
    .le(0.0001)
    .all()
)

financials["roe"] = roe_clean.astype("Float64")

print("ROE 标准化完成")
print("roe dtype:", financials["roe"].dtype)
print("修复的 ROE 记录数:", roe_direct_error_mask.sum())

ROE 标准化完成
roe dtype: Float64
修复的 ROE 记录数: 3


In [64]:
conflict_row_mask = ~non_conflict_mask

conflict_resolution = financials.loc[
    conflict_row_mask,
    [
        "stock_code",
        "year",
        "net_profit",
        "total_assets",
        "total_liabilities",
        "roe",
    ],
].copy()

conflict_resolution["roe_from_current_profit"] = (
    conflict_resolution["net_profit"]
    / (
        conflict_resolution["total_assets"]
        - conflict_resolution["total_liabilities"]
    )
).round(4)

conflict_resolution["roe_gap"] = (
    conflict_resolution["roe"]
    - conflict_resolution["roe_from_current_profit"]
).abs()

conflict_resolution["roe_consistent"] = (
    conflict_resolution["roe_gap"]
    .le(0.0001)
)

conflict_resolution = conflict_resolution.sort_values(
    key_cols + ["roe_gap"]
)

conflict_resolution

,stock_code,year,net_profit,total_assets,total_liabilities,roe,roe_from_current_profit,roe_gap,roe_consistent
10,000002,2024,1.471324e+10,9274052273.0,2248802709.0,2.0943,2.0943,0.0,True
250,000002,2024,1.500893e+09,9274052273.0,2248802709.0,2.0943,0.2136,1.8807,False
27,000005,2023,2.855787e+09,33191213417.0,20487864880.0,0.2248,0.2248,0.0,True
251,000005,2023,1.766409e+09,33191213417.0,20487864880.0,0.2248,0.1391,0.0857,False
44,000008,2022,-3.103548e+09,2729598687.0,701324636.0,-1.5301,-1.5301,0.0,True
252,000008,2022,1.573933e+09,2729598687.0,701324636.0,-1.5301,0.776,2.3061,False
61,000011,2021,5.679525e+08,11495096003.0,5363492081.0,0.0926,0.0926,0.0,True
253,000011,2021,-3.872496e+08,11495096003.0,5363492081.0,0.0926,-0.0632,0.1558,False
78,000014,2020,8.162641e+08,41000762909.0,15785504488.0,0.0324,0.0324,0.0,True
254,000014,2020,1.210936e+09,41000762909.0,15785504488.0,0.0324,0.048,0.0156,False


In [65]:
conflict_resolution_summary = (
    conflict_resolution
    .groupby(key_cols, dropna=False)
    .agg(
        row_count=(
            "net_profit",
            "size",
        ),
        roe_consistent_rows=(
            "roe_consistent",
            "sum",
        ),
        min_roe_gap=(
            "roe_gap",
            "min",
        ),
        max_roe_gap=(
            "roe_gap",
            "max",
        ),
    )
    .reset_index()
)

conflict_resolution_summary

,stock_code,year,row_count,roe_consistent_rows,min_roe_gap,max_roe_gap
0,000002,2024,2,1,0.0,1.8807
1,000005,2023,2,1,0.0,0.0857
2,000008,2022,2,1,0.0,2.3061
3,000011,2021,2,1,0.0,0.1558
4,000014,2020,2,1,0.0,0.0156
5,000016,2025,2,1,0.0,0.1389
6,000019,2024,2,1,0.0,0.348
7,000022,2023,2,1,0.0,0.0884
8,000025,2022,2,1,0.0,0.2402
9,000028,2021,2,1,0.0,0.1952


In [66]:
print(
    "冲突组数:",
    len(conflict_resolution_summary),
)

print(
    "恰好只有一条 ROE 一致记录的组数:",
    conflict_resolution_summary[
        "roe_consistent_rows"
    ].eq(1).sum(),
)

print(
    "无法唯一判断的组数:",
    (
        ~conflict_resolution_summary[
            "roe_consistent_rows"
        ].eq(1)
    ).sum(),
)

冲突组数: 10
恰好只有一条 ROE 一致记录的组数: 10
无法唯一判断的组数: 0


In [67]:
conflict_keep_candidates = conflict_resolution.loc[
    conflict_resolution["roe_consistent"]
].copy()

conflict_drop_candidates = conflict_resolution.loc[
    ~conflict_resolution["roe_consistent"]
].copy()

print(
    "候选保留记录数:",
    len(conflict_keep_candidates),
)

print(
    "候选排除记录数:",
    len(conflict_drop_candidates),
)

候选保留记录数: 10
候选排除记录数: 10


## 第八阶段：解决 net_profit 冲突记录

对剩余 10 个重复 firm-year 组进行内部一致性核验。

这些组中除 `net_profit` 外其他字段均一致，因此利用已确认的 ROE 定义：

`roe = net_profit / (total_assets - total_liabilities)`

分别检验两条记录。

结果显示每个冲突组均恰好有一条记录与 ROE 一致，另一条不一致，
因此本模拟训练数据中保留一致记录，删除不一致副本。

真实科研数据出现类似冲突时，应优先回到原始数据库、年报或数据来源进行核查；
内部一致性检验主要用于定位问题，不应无条件替代源数据核验。

In [68]:
assert len(conflict_resolution_summary) == 10

assert (
    conflict_resolution_summary[
        "row_count"
    ].eq(2).all()
)

assert (
    conflict_resolution_summary[
        "roe_consistent_rows"
    ].eq(1).all()
)

assert len(conflict_keep_candidates) == 10
assert len(conflict_drop_candidates) == 10

print("10 个 conflict 组均可唯一判断")

10 个 conflict 组均可唯一判断


In [69]:
conflict_drop_index = (
    conflict_drop_candidates.index
)

financials_after_conflict_resolution = (
    financials
    .drop(index=conflict_drop_index)
    .copy()
)

print("处理前记录数:", len(financials))
print(
    "删除的冲突副本数:",
    len(conflict_drop_index),
)
print(
    "处理后记录数:",
    len(financials_after_conflict_resolution),
)

处理前记录数: 250
删除的冲突副本数: 10
处理后记录数: 240


In [70]:
remaining_duplicate_mask = (
    financials_after_conflict_resolution
    .duplicated(
        subset=key_cols,
        keep=False,
    )
)

print(
    "处理后重复 firm-year 记录数:",
    remaining_duplicate_mask.sum(),
)

print(
    "处理后完全重复后续记录数:",
    financials_after_conflict_resolution
    .duplicated(keep="first")
    .sum(),
)

print(
    "处理后总记录数:",
    len(financials_after_conflict_resolution),
)

处理后重复 firm-year 记录数: 0
处理后完全重复后续记录数: 0
处理后总记录数: 240


In [71]:
print(
    "年份分布:"
)

print(
    financials_after_conflict_resolution[
        "year"
    ]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\n总记录数:",
    len(financials_after_conflict_resolution),
)

print(
    "不同年份数:",
    financials_after_conflict_resolution[
        "year"
    ].nunique(dropna=True),
)

年份分布:
year
2020    40
2021    40
2022    40
2023    40
2024    40
2025    40
Name: count, dtype: Int64

总记录数: 240
不同年份数: 6


In [72]:
assert len(financials_after_conflict_resolution) == 240

assert (
    financials_after_conflict_resolution
    .duplicated(
        subset=key_cols,
        keep=False,
    )
    .sum()
    == 0
)

assert (
    financials_after_conflict_resolution
    .duplicated(keep="first")
    .sum()
    == 0
)

financials = (
    financials_after_conflict_resolution
)

print("firm-year 冲突处理完成")
print("当前记录数:", len(financials))

firm-year 冲突处理完成
当前记录数: 240


In [73]:
missing_stock_code_rows = financials.loc[
    financials["stock_code"].isna(),
    [
        "stock_code",
        "company_name",
        "year",
    ],
]

print(
    "当前 stock_code 缺失记录数:",
    len(missing_stock_code_rows),
)

missing_stock_code_rows

当前 stock_code 缺失记录数: 1


,stock_code,company_name,year
21,<NA>,中盛科技股份有限公司,2023


In [74]:
company_name_clean = (
    financials["company_name"]
    .astype("string")
    .str.strip()
)

company_name_changed_mask = (
    financials["company_name"]
    .astype("string")
    .ne(company_name_clean)
    .fillna(False)
)

print(
    "公司名称仅因首尾空格发生变化的记录数:",
    company_name_changed_mask.sum(),
)

pd.DataFrame(
    {
        "before": financials.loc[
            company_name_changed_mask,
            "company_name",
        ].map(repr),
        "after": company_name_clean.loc[
            company_name_changed_mask
        ].map(repr),
    }
)

公司名称仅因首尾空格发生变化的记录数: 6


,before,after
1,' 华辰科技股份有限公司 ','华辰科技股份有限公司'
43,' 星海科技股份有限公司 ','星海科技股份有限公司'
85,' 永盛科技股份有限公司 ','永盛科技股份有限公司'
127,' 远望科技股份有限公司 ','远望科技股份有限公司'
169,' 绿能科技股份有限公司 ','绿能科技股份有限公司'
211,' 联盛科技股份有限公司 ','联盛科技股份有限公司'


In [75]:
name_code_pairs = (
    pd.DataFrame(
        {
            "company_name_clean": company_name_clean,
            "stock_code": financials["stock_code"],
        }
    )
    .dropna(
        subset=[
            "company_name_clean",
            "stock_code",
        ]
    )
    .drop_duplicates()
)

name_code_counts = (
    name_code_pairs
    .groupby("company_name_clean")[
        "stock_code"
    ]
    .nunique()
)

print(
    "可唯一对应股票代码的公司名数量:",
    name_code_counts.eq(1).sum(),
)

print(
    "对应多个股票代码的公司名数量:",
    name_code_counts.gt(1).sum(),
)

可唯一对应股票代码的公司名数量: 54
对应多个股票代码的公司名数量: 0


In [76]:
unique_name_code_map = (
    name_code_pairs[
        name_code_pairs[
            "company_name_clean"
        ].isin(
            name_code_counts[
                name_code_counts.eq(1)
            ].index
        )
    ]
    .set_index("company_name_clean")[
        "stock_code"
    ]
)

missing_code_candidates = pd.DataFrame(
    {
        "company_name": company_name_clean.loc[
            financials["stock_code"].isna()
        ],
    }
)

missing_code_candidates[
    "candidate_stock_code"
] = (
    missing_code_candidates[
        "company_name"
    ].map(unique_name_code_map)
)

missing_code_candidates

,company_name,candidate_stock_code
21,中盛科技股份有限公司,000004


In [77]:
print(
    "缺失代码记录数:",
    len(missing_code_candidates),
)

print(
    "可唯一恢复的记录数:",
    missing_code_candidates[
        "candidate_stock_code"
    ].notna().sum(),
)

缺失代码记录数: 1
可唯一恢复的记录数: 1


In [78]:
stock_code_recovered = financials[
    "stock_code"
].copy()

missing_stock_code_mask = (
    stock_code_recovered.isna()
)

stock_code_recovered.loc[
    missing_stock_code_mask
] = (
    company_name_clean.loc[
        missing_stock_code_mask
    ].map(unique_name_code_map)
)

assert (
    stock_code_recovered.isna().sum()
    == 0
)

financials["company_name"] = (
    company_name_clean
)

financials["stock_code"] = (
    stock_code_recovered
)

print(
    "stock_code 剩余缺失:",
    financials["stock_code"].isna().sum(),
)

print(
    "company_name dtype:",
    financials["company_name"].dtype,
)

stock_code 剩余缺失: 0
company_name dtype: string


In [79]:
final_key_duplicate_mask = (
    financials
    .duplicated(
        subset=key_cols,
        keep=False,
    )
)

print(
    "stock_code 缺失数:",
    financials["stock_code"].isna().sum(),
)

print(
    "year 缺失数:",
    financials["year"].isna().sum(),
)

print(
    "重复 firm-year 记录数:",
    final_key_duplicate_mask.sum(),
)

print(
    "不同股票代码数:",
    financials["stock_code"].nunique(),
)

print(
    "总记录数:",
    len(financials),
)

stock_code 缺失数: 0
year 缺失数: 0
重复 firm-year 记录数: 0
不同股票代码数: 40
总记录数: 240


## 第十阶段：剩余缺失值与数值异常审计

在完成 firm-year 主键清洗后，对当前 240 条唯一公司年度记录进行最终数值质量审计。

本阶段重点检查：

- 各变量剩余缺失值；
- 数值变量的分布范围；
- 极端值与潜在错误值；
- 区分“统计上极端但可能真实”与“已确认的数据污染”。

原则上不因数值极端而机械删除记录。

In [80]:
current_quality_summary = pd.DataFrame(
    {
        "dtype": financials.dtypes.astype(str),
        "missing": financials.isna().sum(),
        "unique": financials.nunique(dropna=True),
    }
)

current_quality_summary

,dtype,missing,unique
stock_code,string,0,40
company_name,string,0,54
year,Int64,0,6
total_assets,Float64,0,240
total_liabilities,Float64,1,239
revenue,Float64,0,240
net_profit,float64,1,239
cash,float64,1,239
rd_expense,Float64,1,239
roe,Float64,0,235


In [81]:
numeric_audit_cols = [
    "total_assets",
    "total_liabilities",
    "revenue",
    "net_profit",
    "cash",
    "rd_expense",
    "roe",
    "employees",
]

numeric_distribution = (
    financials[numeric_audit_cols]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.50,
            0.95,
            0.99,
        ]
    )
    .T
)

numeric_distribution

,count,mean,std,min,1%,5%,50%,95%,99%,max
total_assets,240.0,72058518584.25,643737513967.527954,825487570.0,1899693036.42,5023601840.650001,30552434931.0,56871431512.900002,59340521875.139992,9999999999999.0
total_liabilities,239.0,14262074766.506277,10295793166.889219,440373370.0,659600782.44,1580081920.9,11799754463.0,33619231014.199989,42450318214.560013,46579802524.0
revenue,240.0,23019852151.3125,13805259930.435583,416786404.0,936848825.46,2312635203.75,23750994858.0,43633926168.649994,44473323935.580002,44787773048.0
net_profit,239.0,1515154266.698745,3231752706.389039,-9949049437.0,-7370372314.92,-2973324518.9,823332467.0,7443886311.699997,10912883295.460005,14713244863.0
cash,239.0,5368783743.56067,4107987658.040772,209623835.0,293861769.92,645318975.8,4379036316.0,13847565890.799997,16529450187.460001,16781330423.0
rd_expense,239.0,1366511817.644351,1154295388.434731,17999647.0,27008726.34,68227954.5,1078463270.0,3731448606.8,4849254325.240001,4982016458.0
roe,240.0,0.33905,2.570382,-1.9169,-1.288271,-0.314225,0.05245,0.952335,4.3106,38.593
employees,239.0,63675.937238,645561.754047,246.0,888.08,2520.2,20773.0,42608.1,44807.44,9999999.0


In [82]:
for col in numeric_audit_cols:
    print("=" * 70)
    print("变量:", col)

    valid_values = financials[
        ["stock_code", "year", col]
    ].dropna(subset=[col])

    print("\n最小 3 条:")
    print(
        valid_values
        .nsmallest(3, col)
        .to_string(index=False)
    )

    print("\n最大 3 条:")
    print(
        valid_values
        .nlargest(3, col)
        .to_string(index=False)
    )

变量: total_assets

最小 3 条:
stock_code  year  total_assets
    000016  2021   825487570.0
    000035  2021  1489630536.0
    000008  2020  1877325765.0

最大 3 条:
stock_code  year     total_assets
    000028  2024  9999999999999.0
    000006  2020    59612851301.0
    000017  2024    59459915802.0
变量: total_liabilities

最小 3 条:
stock_code  year  total_liabilities
    000024  2021        440373370.0
    000001  2023        476851735.0
    000016  2021        634028098.0

最大 3 条:
stock_code  year  total_liabilities
    000006  2020      46579802524.0
    000020  2020      46173706890.0
    000039  2025      43368882786.0
变量: revenue

最小 3 条:
stock_code  year      revenue
    000032  2022  416786404.0
    000033  2024  497913775.0
    000020  2025  872910309.0

最大 3 条:
stock_code  year        revenue
    000026  2024  44787773048.0
    000021  2021  44665518439.0
    000002  2024  44523089214.0
变量: net_profit

最小 3 条:
stock_code  year    net_profit
    000023  2022 -9.949049e+09
    000030  2

In [83]:
known_numeric_anomaly_mask = (
    financials["total_assets"].eq(9_999_999_999_999)
    | financials["net_profit"].eq(-8_888_888_888)
    | financials["employees"].eq(9_999_999)
)

known_numeric_anomalies = financials.loc[
    known_numeric_anomaly_mask,
    [
        "stock_code",
        "year",
        "total_assets",
        "net_profit",
        "employees",
        "roe",
    ],
].copy()

print(
    "已知注入数值异常记录数:",
    len(known_numeric_anomalies),
)

known_numeric_anomalies

已知注入数值异常记录数: 3


,stock_code,year,total_assets,net_profit,employees,roe
130,000022,2024,6624282171.0,7.056042e+09,9999999.0,1.7754
166,000028,2024,9999999999999.0,-1.937456e+09,18699.0,-0.1328
177,000030,2023,39215609364.0,-8.888889e+09,42032.0,0.0306


## 第十一阶段：已确认数值异常处理

数值分布审计发现 3 条由模拟数据生成器明确注入的异常值：

- `total_assets = 9,999,999,999,999`
- `net_profit = -8,888,888,888`
- `employees = 9,999,999`

这些值属于已确认的数据污染，而不是单纯的统计极端值。

由于无法从当前观测中可靠恢复真实值，本阶段不进行均值填补、公式反推或前后期替代，而是将对应字段设为缺失值，同时保留其余有效字段和整个 firm-year 记录。

真实科研中应优先回到原始数据库或年报核验。

In [84]:
assets_anomaly_mask = (
    financials["total_assets"]
    .eq(9_999_999_999_999)
)

profit_anomaly_mask = (
    financials["net_profit"]
    .eq(-8_888_888_888)
)

employees_anomaly_mask = (
    financials["employees"]
    .eq(9_999_999)
)

print(
    "total_assets 已确认异常:",
    assets_anomaly_mask.sum(),
)

print(
    "net_profit 已确认异常:",
    profit_anomaly_mask.sum(),
)

print(
    "employees 已确认异常:",
    employees_anomaly_mask.sum(),
)

total_assets 已确认异常: 1
net_profit 已确认异常: 1
employees 已确认异常: 1


In [85]:
numeric_anomaly_log = pd.DataFrame(
    [
        {
            "stock_code": financials.loc[
                assets_anomaly_mask,
                "stock_code",
            ].iloc[0],
            "year": financials.loc[
                assets_anomaly_mask,
                "year",
            ].iloc[0],
            "variable": "total_assets",
            "original_value": financials.loc[
                assets_anomaly_mask,
                "total_assets",
            ].iloc[0],
            "action": "set_missing",
            "reason": "known injected anomaly",
        },
        {
            "stock_code": financials.loc[
                profit_anomaly_mask,
                "stock_code",
            ].iloc[0],
            "year": financials.loc[
                profit_anomaly_mask,
                "year",
            ].iloc[0],
            "variable": "net_profit",
            "original_value": financials.loc[
                profit_anomaly_mask,
                "net_profit",
            ].iloc[0],
            "action": "set_missing",
            "reason": "known injected anomaly",
        },
        {
            "stock_code": financials.loc[
                employees_anomaly_mask,
                "stock_code",
            ].iloc[0],
            "year": financials.loc[
                employees_anomaly_mask,
                "year",
            ].iloc[0],
            "variable": "employees",
            "original_value": financials.loc[
                employees_anomaly_mask,
                "employees",
            ].iloc[0],
            "action": "set_missing",
            "reason": "known injected anomaly",
        },
    ]
)

numeric_anomaly_log

,stock_code,year,variable,original_value,action,reason
0,000028,2024,total_assets,1.000000e+13,set_missing,known injected anomaly
1,000030,2023,net_profit,-8.888889e+09,set_missing,known injected anomaly
2,000022,2024,employees,9.999999e+06,set_missing,known injected anomaly


In [86]:
financials_after_anomaly_handling = (
    financials.copy()
)

financials_after_anomaly_handling.loc[
    assets_anomaly_mask,
    "total_assets",
] = pd.NA

financials_after_anomaly_handling.loc[
    profit_anomaly_mask,
    "net_profit",
] = pd.NA

financials_after_anomaly_handling.loc[
    employees_anomaly_mask,
    "employees",
] = pd.NA

In [87]:
anomaly_handling_check = pd.DataFrame(
    {
        "before_missing": financials.isna().sum(),
        "after_missing": (
            financials_after_anomaly_handling
            .isna()
            .sum()
        ),
    }
)

anomaly_handling_check[
    [
        "before_missing",
        "after_missing",
    ]
]

,before_missing,after_missing
stock_code,0,0
company_name,0,0
year,0,0
total_assets,0,1
total_liabilities,1,1
revenue,0,0
net_profit,1,2
cash,1,1
rd_expense,1,1
roe,0,0


In [88]:
print(
    "剩余异常 total_assets:",
    financials_after_anomaly_handling[
        "total_assets"
    ].eq(9_999_999_999_999).sum(),
)

print(
    "剩余异常 net_profit:",
    financials_after_anomaly_handling[
        "net_profit"
    ].eq(-8_888_888_888).sum(),
)

print(
    "剩余异常 employees:",
    financials_after_anomaly_handling[
        "employees"
    ].eq(9_999_999).sum(),
)

剩余异常 total_assets: 0
剩余异常 net_profit: 0
剩余异常 employees: 0


In [89]:
assert assets_anomaly_mask.sum() == 1
assert profit_anomaly_mask.sum() == 1
assert employees_anomaly_mask.sum() == 1

assert (
    financials_after_anomaly_handling[
        "total_assets"
    ].eq(9_999_999_999_999).sum()
    == 0
)

assert (
    financials_after_anomaly_handling[
        "net_profit"
    ].eq(-8_888_888_888).sum()
    == 0
)

assert (
    financials_after_anomaly_handling[
        "employees"
    ].eq(9_999_999).sum()
    == 0
)

financials = (
    financials_after_anomaly_handling
)

print("已确认数值污染处理完成")
print("当前记录数:", len(financials))

已确认数值污染处理完成
当前记录数: 240


In [90]:
missing_after_cleaning = (
    financials
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_after_cleaning

,missing_count
stock_code,0
company_name,0
year,0
total_assets,1
total_liabilities,1
revenue,0
net_profit,2
cash,1
rd_expense,1
roe,0


## 第十二阶段：清洗结果总体验收

在输出 processed 数据前，对当前财务面板进行最终质量检查。

验收内容包括：

- 数据类型是否规范；
- firm-year 主键是否完整且唯一；
- 面板规模是否符合预期；
- 数值字段是否仍存在字符串残留；
- 缺失值数量是否与清洗记录一致；
- 已确认异常值是否已经移除；
- 原始数据仍保持不修改。

验收通过后，才输出标准化 processed 数据。

In [91]:
money_cols = [
    "total_assets",
    "total_liabilities",
    "revenue",
    "net_profit",
    "cash",
    "rd_expense",
]

for col in money_cols:
    financials[col] = financials[col].astype("Float64")

financials["roe"] = financials["roe"].astype("Float64")

employees_non_missing = (
    financials["employees"]
    .dropna()
)

assert (
    employees_non_missing
    .mod(1)
    .eq(0)
    .all()
)

financials["employees"] = (
    financials["employees"]
    .astype("Int64")
)

financials.dtypes

stock_code            string
company_name          string
year                   Int64
total_assets         Float64
total_liabilities    Float64
revenue              Float64
net_profit           Float64
cash                 Float64
rd_expense           Float64
roe                  Float64
employees              Int64
dtype: object

In [92]:
final_numeric_cols = [
    "total_assets",
    "total_liabilities",
    "revenue",
    "net_profit",
    "cash",
    "rd_expense",
    "roe",
    "employees",
]

numeric_string_residue = pd.Series(
    {
        col: financials[col]
        .map(lambda x: isinstance(x, str))
        .sum()
        for col in final_numeric_cols
    },
    name="string_count",
)

numeric_string_residue

total_assets         0
total_liabilities    0
revenue              0
net_profit           0
cash                 0
rd_expense           0
roe                  0
employees            0
Name: string_count, dtype: int64

In [93]:
panel_validation = {
    "rows": len(financials),
    "unique_stock_codes": financials[
        "stock_code"
    ].nunique(),
    "unique_years": financials[
        "year"
    ].nunique(),
    "missing_stock_code": financials[
        "stock_code"
    ].isna().sum(),
    "missing_year": financials[
        "year"
    ].isna().sum(),
    "duplicate_firm_year_rows": (
        financials
        .duplicated(
            subset=key_cols,
            keep=False,
        )
        .sum()
    ),
}

pd.Series(panel_validation)

rows                        240
unique_stock_codes           40
unique_years                  6
missing_stock_code            0
missing_year                  0
duplicate_firm_year_rows      0
dtype: int64

In [94]:
year_counts_final = (
    financials["year"]
    .value_counts()
    .sort_index()
)

year_counts_final

year
2020    40
2021    40
2022    40
2023    40
2024    40
2025    40
Name: count, dtype: Int64

In [95]:
expected_missing = pd.Series(
    {
        "stock_code": 0,
        "company_name": 0,
        "year": 0,
        "total_assets": 1,
        "total_liabilities": 1,
        "revenue": 0,
        "net_profit": 2,
        "cash": 1,
        "rd_expense": 1,
        "roe": 0,
        "employees": 2,
    },
    name="expected_missing",
)

actual_missing = (
    financials
    .isna()
    .sum()
    .rename("actual_missing")
)

missing_validation = pd.concat(
    [
        expected_missing,
        actual_missing,
    ],
    axis=1,
)

missing_validation["match"] = (
    missing_validation[
        "expected_missing"
    ]
    == missing_validation[
        "actual_missing"
    ]
)

missing_validation

,expected_missing,actual_missing,match
stock_code,0,0,True
company_name,0,0,True
year,0,0,True
total_assets,1,1,True
total_liabilities,1,1,True
revenue,0,0,True
net_profit,2,2,True
cash,1,1,True
rd_expense,1,1,True
roe,0,0,True


In [96]:
known_anomaly_validation = pd.Series(
    {
        "bad_total_assets_remaining": (
            financials["total_assets"]
            .eq(9_999_999_999_999)
            .sum()
        ),
        "bad_net_profit_remaining": (
            financials["net_profit"]
            .eq(-8_888_888_888)
            .sum()
        ),
        "bad_employees_remaining": (
            financials["employees"]
            .eq(9_999_999)
            .sum()
        ),
    }
)

known_anomaly_validation

bad_total_assets_remaining    0
bad_net_profit_remaining      0
bad_employees_remaining       0
dtype: int64

In [97]:
assert len(financials) == 240

assert financials["stock_code"].nunique() == 40
assert financials["year"].nunique() == 6

assert financials["stock_code"].isna().sum() == 0
assert financials["year"].isna().sum() == 0

assert (
    financials
    .duplicated(
        subset=key_cols,
        keep=False,
    )
    .sum()
    == 0
)

assert year_counts_final.eq(40).all()

assert numeric_string_residue.eq(0).all()

assert missing_validation["match"].all()

assert known_anomaly_validation.eq(0).all()

print("财务面板最终质量验收通过")
print("记录数:", len(financials))
print(
    "firm-year 主键数:",
    financials[key_cols]
    .drop_duplicates()
    .shape[0],
)

财务面板最终质量验收通过
记录数: 240
firm-year 主键数: 240


## 第十三阶段：输出标准化 processed 数据

在最终质量验收通过后，输出标准化财务面板数据。

输出两种格式：

- Parquet：作为 Python 数据工程侧的标准 processed 数据；
- Stata `.dta`：作为后续 Stata 实证分析的直接输入文件。

两种格式均由同一份清洗后的数据生成，避免 Python 与 Stata 分别维护不同的数据清洗逻辑。

输出前按 `stock_code + year` 排序并重置行索引。

In [98]:
DATA_PROCESSED.mkdir(
    parents=True,
    exist_ok=True,
)

financials_final = (
    financials
    .sort_values(key_cols)
    .reset_index(drop=True)
    .copy()
)

parquet_path = (
    DATA_PROCESSED
    / "firm_financials_clean.parquet"
)

stata_path = (
    DATA_PROCESSED
    / "firm_financials_clean.dta"
)

# Python 标准版本
financials_final.to_parquet(
    parquet_path,
    index=False,
)

# Stata 导出版本
stata_export = financials_final.copy()

stata_export["stock_code"] = (
    stata_export["stock_code"]
    .astype(str)
)

stata_export["company_name"] = (
    stata_export["company_name"]
    .astype(str)
)

stata_export["year"] = (
    stata_export["year"]
    .astype("int16")
)

for col in money_cols + ["roe"]:
    stata_export[col] = (
        stata_export[col]
        .astype("float64")
    )

stata_export["employees"] = (
    stata_export["employees"]
    .astype("float64")
)

stata_export.to_stata(
    stata_path,
    write_index=False,
    version=118,
)

print(
    "Parquet 输出:",
    parquet_path.name,
)

print(
    "Stata 输出:",
    stata_path.name,
)

Parquet 输出: firm_financials_clean.parquet
Stata 输出: firm_financials_clean.dta


In [99]:
parquet_check = pd.read_parquet(
    parquet_path
)

assert len(parquet_check) == 240

assert (
    parquet_check["stock_code"]
    .astype("string")
    .str.fullmatch(r"\d{6}")
    .all()
)

assert (
    parquet_check
    .duplicated(
        subset=key_cols,
        keep=False,
    )
    .sum()
    == 0
)

assert (
    parquet_check["year"]
    .value_counts()
    .sort_index()
    .eq(40)
    .all()
)

assert (
    parquet_check
    .isna()
    .sum()
    .eq(actual_missing)
    .all()
)

print("Parquet 回读验证通过")
print("shape:", parquet_check.shape)

parquet_check.dtypes

Parquet 回读验证通过
shape: (240, 11)


stock_code            string
company_name          string
year                   Int64
total_assets         Float64
total_liabilities    Float64
revenue              Float64
net_profit           Float64
cash                 Float64
rd_expense           Float64
roe                  Float64
employees              Int64
dtype: object

In [100]:
stata_check = pd.read_stata(
    stata_path,
    convert_categoricals=False,
)

assert len(stata_check) == 240

assert (
    stata_check["stock_code"]
    .str.fullmatch(r"\d{6}")
    .all()
)

assert (
    stata_check
    .duplicated(
        subset=key_cols,
        keep=False,
    )
    .sum()
    == 0
)

assert (
    stata_check["stock_code"]
    .nunique()
    == 40
)

assert (
    stata_check["year"]
    .value_counts()
    .sort_index()
    .eq(40)
    .all()
)

print("Stata .dta 回读验证通过")
print("shape:", stata_check.shape)

stata_check.dtypes

Stata .dta 回读验证通过
shape: (240, 11)


stock_code               str
company_name             str
year                   int16
total_assets         float64
total_liabilities    float64
revenue              float64
net_profit           float64
cash                 float64
rd_expense           float64
roe                  float64
employees            float64
dtype: object

In [101]:
parquet_keys = set(
    map(
        tuple,
        parquet_check[
            key_cols
        ].to_numpy(),
    )
)

stata_keys = set(
    map(
        tuple,
        stata_check[
            key_cols
        ].to_numpy(),
    )
)

print(
    "Parquet firm-year 数:",
    len(parquet_keys),
)

print(
    "Stata firm-year 数:",
    len(stata_keys),
)

print(
    "两种格式主键完全一致:",
    parquet_keys == stata_keys,
)

Parquet firm-year 数: 240
Stata firm-year 数: 240
两种格式主键完全一致: True


## 本阶段最终结论

本 Notebook 已完成模拟公司年度财务面板的第一轮正式清洗。

最终结果：

- 原始数据共 260 条记录；
- 删除 10 条完全重复副本；
- 对 10 个冲突 firm-year 组进行内部一致性核验，并删除 10 条已确认错误的冲突副本；
- 恢复 1 条缺失股票代码；
- 最终得到 240 个唯一 firm-year，对应 40 家公司 × 2020–2025 共 6 年；
- 股票代码统一为 6 位字符串；
- 年份统一为整数类型；
- 数值字段中的千位逗号和伪缺失值已标准化；
- 3 条已确认错误的 ROE 已修复；
- 与财务关系一致的极端 ROE 保留；
- 3 个已确认注入的数值污染单元格被设为缺失，而没有删除整条 firm-year；
- 剩余缺失值保留，不进行无依据均值填补。

最终数据已分别输出为 Parquet 和 Stata `.dta` 格式，并完成：

- Python 回读验证；
- 两种格式主键一致性验证；
- Stata/MP 实际读取验证；
- `stock_code + year` 唯一性验证；
- 年份分布和缺失值结构验证。

原始 raw 数据始终未被修改。

后续将继续清洗公司基本信息和专利数据，并逐步构建完整研究面板。